# 4. Modelling

## 4.1 Split Data in Training and Testing Set
Chronologically split the dataset: 80% for training, 20% for testing. Use TimeSeriesSplit -  for cross-validation to preserve order.
## 4.2 Select Suitable Modeling Techniques
 - Logistic Regression: interpretable baseline
 - Random Forest: captures non-linear feature interactions
 - SVM: non-linear classification for moderate-size datasets
## 4.3 Generate Test Design
Use sequential cross-validation folds; evaluate accuracy, precision, recall, F1-score. Features include BTC/NQ returns, volatilities, rolling correlations, and time-of-day indicators.
## 4.4 Apply Selected Modeling Techniques
Train models on training folds (standardize for LR & SVM!!!). Validate using cross-validation and record fold-wise performance.
## 4.5 Assess Model
Compare models based on cross-validation metrics; examine Random Forest feature importance to identify key predictive factors.

# 5. Evaluation

## 5.1 Apply Test Set
Test final models on the held-out set to evaluate real-world predictive performance.
## 5.2 Interpret Result
## 5.3 Cross-Check Results
## 5.4 Final Model Training
## 5.5 Review Process
Consider additional features, hyperparameter tuning, and alternative models for further optimization.
## 5.6 Interpret Results
Assess reduction in unprofitable trades, improved signal filtering, and enhanced risk-adjusted returns over the baseline strategy.

# Analytical Feature Engineering

**Goal:** to transform data into model-ready analytical table


#### 1.1 Cross-Market Features

 - BTC_return: 1-minute log return of BTC

 - NQ_return: 1-minute log return of NQ (optional for supervised learning if using lagged returns)

 - Rolling Correlation (60-min): correlation between BTC and NQ returns over the last 60 minutes

 - Agreement / Divergence: directional match between BTC and NQ candles (1 if same direction, 0 otherwise)

#### 1.2 Technical Indicators

 - ATR (Average True Range): volatility of NQ over the last N minutes (e.g., 14)

 - Rolling Standard Deviation of Returns: short-term volatility over multiple windows (e.g., 5, 20, 60 minutes)

 - MA (Simple Moving Average): compute SMA for multiple windows (e.g., 5, 20, 60)

 - Include derived features:
Price – SMA
SMA Slope = current SMA – previous SMA
Bollinger Bands: SMA ± 2 × rolling std
 - Derived features:
Distance from upper band = (Price – Upper Band) / (Upper – Lower Band)
Distance from lower band = (Price – Lower Band) / (Upper – Lower Band)
Bandwidth = (Upper – Lower Band) / SMA

#### 1.3 Time & Microstructure Features
 - Hour of the day (0–23) — captures intraday patterns (“golden hours”)
 - Minute of the hour (0–59) — optional fine-grained timing
 - Previous returns / lagged returns — e.g., NQ_return_t-1, NQ_return_t-2
 - Previous cross-market signals — e.g., agreement_t-1, divergence_t-1


In [26]:
import pandas as pd
import numpy as np

# Load final dataset
final_path = "Datasets/final_dataset.csv"
data = pd.read_csv(final_path)

print("Dataset loaded successfully!")
print("Total rows:", len(data))
print("Columns:", data.columns)

# ==============================
# Ensure datetime index
# ==============================
# Convert to datetime with UTC parsing
data['timestamp'] = pd.to_datetime(data['timestamp'], utc=True, errors='coerce')  # parse as UTC

# Remove timezone to make naive datetime
data['timestamp'] = data['timestamp'].dt.tz_convert(None)

# Set timestamp as index
data.set_index('timestamp', inplace=True)

# Extract hour and minute
data['hour'] = data.index.hour
data['minute'] = data.index.minute

print("Datetime index set successfully!")
print(data.index[:5])


# ==============================
# 1.1 Cross-Market Features
# ==============================
data['BTC_return'] = np.log(data['BTC_close'] / data['BTC_close'].shift(1))
data['NQ_return'] = np.log(data['NQ_close'] / data['NQ_close'].shift(1))

# Rolling correlation (60-min)
data['BTC_NQ_corr_60'] = data['BTC_return'].rolling(window=60).corr(data['NQ_return'])

# Agreement / Divergence
data['direction_match'] = ((data['BTC_return'] > 0) & (data['NQ_return'] > 0)) | \
                          ((data['BTC_return'] < 0) & (data['NQ_return'] < 0))
data['direction_match'] = data['direction_match'].astype(int)

# ==============================
# 1.2 Technical Indicators
# ==============================
high = data['NQ_high']
low = data['NQ_low']
close = data['NQ_close']

# ATR
data['H-L'] = high - low
data['H-PC'] = abs(high - close.shift(1))
data['L-PC'] = abs(low - close.shift(1))
data['TR'] = data[['H-L','H-PC','L-PC']].max(axis=1)
data['ATR_14'] = data['TR'].rolling(14).mean()

# Rolling Std of NQ returns
for window in [5, 20, 60]:
    data[f'NQ_ret_std_{window}'] = data['NQ_return'].rolling(window).std()

# SMA, SMA slope, Price minus SMA
for window in [5, 20, 60]:
    data[f'SMA_{window}'] = data['NQ_close'].rolling(window).mean()
    data[f'SMA_{window}_slope'] = data[f'SMA_{window}'] - data[f'SMA_{window}'].shift(1)
    data[f'Price_minus_SMA_{window}'] = data['NQ_close'] - data[f'SMA_{window}']

# Bollinger Bands (20-period)
sma_20 = data['SMA_20']
std_20 = data['NQ_ret_std_20']
data['BB_upper'] = sma_20 + 2 * std_20
data['BB_lower'] = sma_20 - 2 * std_20
data['BB_dist_upper'] = (data['NQ_close'] - data['BB_upper']) / (data['BB_upper'] - data['BB_lower'])
data['BB_dist_lower'] = (data['NQ_close'] - data['BB_lower']) / (data['BB_upper'] - data['BB_lower'])
data['BB_bandwidth'] = (data['BB_upper'] - data['BB_lower']) / sma_20

# ==============================
# 1.3 Lagged / Microstructure Features
# ==============================
for lag in [1, 2, 3]:
    data[f'NQ_return_lag_{lag}'] = data['NQ_return'].shift(lag)
    data[f'direction_match_lag_{lag}'] = data['direction_match'].shift(lag)

# Drop intermediate columns
data.drop(columns=['H-L', 'H-PC', 'L-PC', 'TR'], inplace=True)

Dataset loaded successfully!
Total rows: 72195
Columns: Index(['timestamp', 'BTC_open', 'BTC_high', 'BTC_low', 'BTC_close',
       'BTC_volume', 'NQ_open', 'NQ_high', 'NQ_low', 'NQ_close', 'NQ_volume',
       'BTC_return', 'NQ_return', 'BTC_vol', 'NQ_vol', 'BTC_dir', 'NQ_dir',
       'agreement', 'divergence', 'Hour', 'Minute'],
      dtype='object')
Datetime index set successfully!
DatetimeIndex(['2025-10-01 06:00:00', '2025-10-01 06:01:00',
               '2025-10-01 06:02:00', '2025-10-01 06:03:00',
               '2025-10-01 06:04:00'],
              dtype='datetime64[ns]', name='timestamp', freq=None)


In [27]:
# ==============================
# Save analytical dataset
# ==============================
analytical_path = "Datasets/analytical_dataset.csv"
data.to_csv(analytical_path)
print(f"Analytical dataset saved to {analytical_path}")
print("Shape after feature engineering:", data.shape)


Analytical dataset saved to Datasets/analytical_dataset.csv
Shape after feature engineering: (72195, 48)


# Target Construction & Prediction Framework

## Objective

The objective of this stage is to define a **clear, forward-looking target variable**.
---

## Prediction Horizon

The model operates at a **1-minute frequency**.  
At each time step $ t $, the model has access only to information available up to that moment and is tasked with predicting the **direction of the market at time $ t+1 $**.

This formulation ensures:
- Causal consistency
- No information leakage
- Compatibility with real-time trading systems


## Target Definition

Let $ P_t $ denote the NQ close price at time $ t $.

The **one-minute forward log return** is defined as:

$$
r_{t+1} = \log\left(\frac{P_{t+1}}{P_t}\right)
$$

The binary target variable is constructed as:

$$
y_t =
\begin{cases}
1 & \text{if } r_{t+1} > 0 \\
0 & \text{otherwise}
\end{cases}
$$

Where:
- **1** indicates an upward price movement in the next minute
- **0** indicates a flat or downward movement

This formulation transforms the problem into a **binary classification task**.

---

## Rationale for Using $ t+1 $ as Target

Using the next-period return $ t+1 $ is a standard approach in financial machine learning due to the following reasons:

- Prevents look-ahead bias
- Ensures that predictions are genuinely forward-looking
- Aligns with market microstructure at high frequencies
- Provides a well-defined and interpretable learning objective

Alternative horizons (e.g. $ t+5 $, $ t+10 $) were not used to avoid unnecessary complexity and signal dilution.

---

## Baseline Strategy as Benchmark

A **naïve momentum-based baseline** is used as a reference model:

- If the current return is positive → predict upward movement
- If the current return is negative → predict downward movement

The baseline does **not** define the target, but rather serves as a **benchmark** against which more sophisticated models are evaluated.

This allows for a meaningful comparison and helps assess whether engineered features and machine learning models provide predictive value beyond simple heuristics.

---

## Data Alignment

Because the target depends on $ t+1 $, the final observation in the dataset cannot be labeled and is therefore removed. All remaining observations are properly aligned such that:

- Features are computed using information available at time $ t $
- The target represents the market outcome at time $ t+1 $

This guarantees a clean and realistic supervised learning setup.

In [31]:
import numpy as np
import pandas as pd

# ==============================
# Load analytical dataset
# ==============================
analytical_path = "Datasets/analytical_dataset.csv"
data = pd.read_csv(analytical_path)

# Ensure timestamp index
data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
data.set_index('timestamp', inplace=True)

print("Dataset loaded")
print("Initial shape:", data.shape)

# Columns to ignore when dropping NaNs - 
ignore_nan_cols = ['BB_dist_upper', 'BB_dist_lower', 'BTC_NQ_corr_60']

# Had to do this - because If BB_upper - BB_lower == 0 (i.e., the Bollinger Band width is zero, which can happen when the rolling standard deviation is zero), this creates NaN values.

# Drop rows with NaNs in all other columns
rows_before = len(data)
data = data.dropna(subset=[col for col in data.columns if col not in ignore_nan_cols])
rows_after = len(data)
print(f"Rows removed due to NaNs (ignoring BB_dist_upper/lower): {rows_before - rows_after}")

# Fill remaining NaNs (BB_dist_upper / BB_dist_lower) with 0
data[ignore_nan_cols] = data[ignore_nan_cols].fillna(0)


# ==============================
# Target Construction (t+1)
# ==============================

# Forward 1-minute log return
data['NQ_return_t_plus_1'] = np.log(
    data['NQ_close'].shift(-1) / data['NQ_close']
)

# Binary classification target
data['target'] = (data['NQ_return_t_plus_1'] > 0).astype(int)

# ==============================
# Clean dataset
# ==============================

# Drop last row (no t+1 available)
data = data.iloc[:-1]


print("Final shape after target construction:", data.shape)
print("Target distribution:")
print(data['target'].value_counts(normalize=True))


Dataset loaded
Initial shape: (72195, 48)
Rows removed due to NaNs (ignoring BB_dist_upper/lower): 60
Final shape after target construction: (72134, 50)
Target distribution:
target
0    0.511423
1    0.488577
Name: proportion, dtype: float64


In [32]:
# Save model-ready dataset
model_data_path = "Datasets/model_ready_dataset.csv"
data.to_csv(model_data_path)

print(f"Model-ready dataset saved to {model_data_path}")

Model-ready dataset saved to Datasets/model_ready_dataset.csv


## 2. Chronological Split & Feature Scaling

After constructing the target (`t+1` NQ return) and cleaning the dataset, we split the dataset chronologically to prevent **look-ahead bias**.

### 2.1 Chronological Split

- **Training set:** first 60% of the dataset  
- **Validation set:** next 20%  
- **Test set:** last 20%  

This ensures that the model is always tested on future unseen data.

### 2.2 Feature Scaling

We normalize all input features (except the target columns) using **StandardScaler**:

- StandardScaler subtracts the **mean** and divides by the **standard deviation** of each feature.
- The scaler is **fitted on the training set** only, then applied to validation and test sets to avoid data leakage.

This step ensures that all features are on a comparable scale, which improves model convergence and performance.


In [35]:
from sklearn.preprocessing import StandardScaler

# Chronological split: 60/20/20
n = len(data)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_data = data.iloc[:train_end].copy()
val_data   = data.iloc[train_end:val_end].copy()
test_data  = data.iloc[val_end:].copy()

print("\nDataset shapes:")
print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

# Feature scaling

# Exclude target columns from scaling
target_cols = ['NQ_return_t_plus_1', 'target']
feature_cols = [col for col in data.columns if col not in target_cols]

scaler = StandardScaler()

# Fit on train, transform on train/val/test
train_data[feature_cols] = scaler.fit_transform(train_data[feature_cols])
val_data[feature_cols] = scaler.transform(val_data[feature_cols])
test_data[feature_cols] = scaler.transform(test_data[feature_cols])

print("\nFeature scaling completed.")


Dataset shapes:
Train: (43280, 50)
Validation: (14427, 50)
Test: (14427, 50)

Feature scaling completed.


In [38]:
# ======================================
# Feature Selection for Logistic Regression / SVM
# ======================================

import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ------------------------------
# 1. Define feature set
# ------------------------------
drop_cols = ['target']  # we already have the target
X_train = train_data.drop(columns=drop_cols)
y_train = train_data['target']

X_val = val_data.drop(columns=drop_cols)
y_val = val_data['target']

X_test = test_data.drop(columns=drop_cols)
y_test = test_data['target']

print("Initial feature count:", X_train.shape[1])

# ------------------------------
# 2. Step 1: Variance Threshold
# ------------------------------
selector_var = VarianceThreshold(threshold=1e-5)
X_train_var = selector_var.fit_transform(X_train)
features_var = X_train.columns[selector_var.get_support()]
print("After Variance Threshold, features:", len(features_var))

# Apply to val/test
X_val_var = X_val[features_var]
X_test_var = X_test[features_var]

# ------------------------------
# 3. Step 2: Spearman Correlation Filter
# ------------------------------
X_train_spearman = pd.DataFrame(X_train_var, columns=features_var, index=X_train.index)
corr_matrix = X_train_spearman.corr(method='spearman').abs()

# Remove one feature from each pair with correlation > 0.85
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
print("Features dropped due to high Spearman correlation:", to_drop)

X_train_corr = X_train_spearman.drop(columns=to_drop)
X_val_corr = X_val_var[X_train_corr.columns]
X_test_corr = X_test_var[X_train_corr.columns]

print("Shape after correlation filter:", X_train_corr.shape)

# ------------------------------
# 4. Step 3: Elastic Net for embedded feature selection
# ------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_corr)
X_val_scaled   = scaler.transform(X_val_corr)
X_test_scaled  = scaler.transform(X_test_corr)

# Elastic Net Logistic Regression
enet = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5,
                          max_iter=5000, random_state=42)
enet.fit(X_train_scaled, y_train)

# Select features based on coefficients
sfm = SelectFromModel(enet, prefit=True)
X_train_final = sfm.transform(X_train_scaled)
X_val_final   = sfm.transform(X_val_scaled)
X_test_final  = sfm.transform(X_test_scaled)

selected_features = X_train_corr.columns[sfm.get_support()]
print("Final selected features (Elastic Net):")
print(list(selected_features))
print("Final shape:", X_train_final.shape)


Initial feature count: 49
After Variance Threshold, features: 48
Features dropped due to high Spearman correlation: ['BTC_high', 'BTC_low', 'BTC_close', 'NQ_high', 'NQ_low', 'NQ_close', 'BTC_dir', 'NQ_dir', 'divergence', 'minute', 'direction_match', 'ATR_14', 'NQ_ret_std_20', 'NQ_ret_std_60', 'SMA_5', 'SMA_20', 'SMA_60', 'BB_upper', 'BB_lower', 'BB_dist_upper', 'BB_dist_lower', 'BB_bandwidth']
Shape after correlation filter: (43280, 26)
Final selected features (Elastic Net):
['NQ_open', 'BTC_return', 'NQ_return', 'SMA_5_slope', 'Price_minus_SMA_5', 'Price_minus_SMA_20', 'SMA_60_slope', 'Price_minus_SMA_60', 'NQ_return_lag_1', 'direction_match_lag_3']
Final shape: (43280, 10)


# Predictive Feature Engineering

**Goal:** increase predictive power

__ train thansformations only ot training data -> then apply to test data